# TCS Stock: Linear Regression — OHLCV vs OHLCV + Indicators
**Target:** Next_Day_Close = Close.shift(-1) | **Split:** 80/20 time-based | **Dataset:** TCS.csv (2005–2026)

**Structure:**
1. Load data → EDA → Visualizations
2. Part A: Linear Regression on OHLCV Only (5 features)
3. Part B: Linear Regression on OHLCV + Indicators (14 features)
4. Part C: Compare A vs B
5. Part D: Overfitting / Underfitting Check

## Step 0: Data Collection — Download TCS Stock Data from Yahoo Finance

**Source:** Yahoo Finance via `yfinance` library
**Stock:** TCS.NS (TCS listed on NSE — National Stock Exchange, India)
**Period:** 2005-01-01 to 2026-09-24
**Saved as:** TCS.csv

This is the raw data collection step. The downloaded CSV contains:
Open, High, Low, Close, Volume — daily historical data adjusted for stock splits and dividends.

In [1]:
pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
yf.download('TCS.NS', start='2005-01-01', end='2026-09-24').to_csv('TCS.csv')
print("TCS.csv downloaded and saved successfully!")

[*********************100%***********************]  1 of 1 completed

TCS.csv downloaded and saved successfully!


---
## Step 1: Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

print("Libraries imported successfully!")

Libraries imported successfully!


---
## Step 2: Load Dataset
`TCS.csv` has 2 extra yfinance header rows — skip with `skiprows=[1,2]`. Rename 'Price' → 'Date'.

In [57]:
df = pd.read_csv('TCS.csv', skiprows=[1, 2])
df.rename(columns={'Price': 'Date'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

for col in ['Open','High','Low','Close','Volume']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)
print("Dataset loaded!")
print(f"Rows: {len(df)}")

Dataset loaded!
Rows: 5363


---
## Step 3: Dataset Overview

In [5]:
print(f"Shape     : {df.shape}")
print(f"Date Range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Columns   : {list(df.columns)}")

Shape     : (5363, 5)
Date Range: 2005-01-03 to 2026-09-23
Columns   : ['Open', 'High', 'Low', 'Close', 'Volume']


In [6]:
print("Column Names and Data Types:")
print(df.dtypes)

Column Names and Data Types:
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object


In [7]:
print("First 5 Rows:")
df.head()

First 5 Rows:


,Open,High,Low,Close,Volume
Date,,,,,
2005-01-03,106.848537,108.547923,106.848537,108.272697,4734168
2005-01-04,108.011852,108.331887,106.664507,107.312576,4979104
2005-01-05,107.227811,107.227811,102.435290,106.011681,9604320
2005-01-06,105.611583,107.691811,103.411342,105.648384,8006888
2005-01-07,106.411726,107.611858,105.024376,107.208611,7623136


In [8]:
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"Total missing: {df.isnull().sum().sum()}")

Missing Values per Column:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64
Total missing: 0


In [9]:
print(f"Duplicate Rows: {df.duplicated().sum()}")

Duplicate Rows: 0


In [10]:
print("Statistical Summary:")
df[['Open','High','Low','Close','Volume']].describe().round(2)

Statistical Summary:


,Open,High,Low,Close,Volume
count,5363.00,5363.00,5363.00,5363.00,5363.00
mean,1344.44,1357.71,1330.29,1343.67,3350520.35
std,1188.95,1199.08,1178.30,1188.29,2819411.07
min,75.80,78.58,69.43,75.49,0.00
25%,274.27,277.70,270.58,274.01,1853218.00
50%,959.55,967.05,951.04,958.63,2650784.00
75%,2383.67,2414.17,2349.66,2376.28,3987019.00
max,4251.38,4266.48,4191.92,4230.71,88067154.00
